# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# Show available record sets and their @id fields
record_sets = sorted(dataset.metadata.record_sets, key=lambda o: o.id)
print("Available Record Sets:")
for rs in record_sets:
    print(f"  - @id: {rs.id}; name: {rs.name}; description: {rs.description}")
    print("    Fields and columns (by @id):")
    # Fields
    for field in sorted(rs.fields, key=lambda f: f.id):
        print(f"      field: @id={field.id} (name: {field.name}, dataType: {field.data_type})")
    # Columns
    if hasattr(rs, 'columns') and rs.columns:
        for col in sorted(rs.columns, key=lambda c: c.id):
            print(f"      column: @id={col.id} (name: {col.name})")
    print()
# Optional: preview a single record from each set
for rs in record_sets:
    print(f"Sample record from RecordSet '@id={rs.id}':")
    records = list(dataset.records(record_set=rs.id))
    if records:
        print(records[0])
    else:
        print("  (no records)")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# Extract data from each record set
# Use all record_sets found above
record_sets_ids = [rs.id for rs in dataset.metadata.record_sets]
dataframes = {}

for record_set_id in record_sets_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df

# Preview available columns for each record set
for rsid in dataframes:
    print(f"Columns for RecordSet @id={rsid}: {dataframes[rsid].columns.tolist()}")
# Pick one for further EDA:
selected_record_set_id = record_sets_ids[0] if record_sets_ids else None
if selected_record_set_id:
    print(f"Head of DataFrame for {selected_record_set_id}:")
    display(dataframes[selected_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes to prepare it for further analysis.

In [ ]:
# Choose a record set and numeric field for analysis
import numpy as np
if not dataframes:
    print("No dataframes loaded.")
else:
    # Select the largest DataFrame
    selected_record_set_id = max(dataframes, key=lambda k: dataframes[k].shape[0])
    df = dataframes[selected_record_set_id]

    # Try to auto-detect a numeric field
    numeric_fields = df.select_dtypes(include=[np.number]).columns.tolist()
    if numeric_fields:
        numeric_field_id = numeric_fields[0]
        print(f"Using numeric field for analysis: {numeric_field_id}")

        threshold = df[numeric_field_id].mean()
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
        print(filtered_df.head())

        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Find a candidate for grouping (categorical string field)
        string_fields = df.select_dtypes(include=['object', 'category']).columns.tolist()
        group_field = None
        for c in string_fields:
            if df[c].nunique() < max(5, min(50, len(df)//10)):
                group_field = c
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field_id].mean()
            print(f"\nGrouped data by {group_field} (showing mean {numeric_field_id}):")
            print(grouped_df)
        else:
            print("No suitable group field found.")
    else:
        print("No numeric fields for EDA.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if dataframes and selected_record_set_id and numeric_fields:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id], bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    # If grouping field was found
    if group_field is not None:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field}")
        plt.show()
else:
    print("No numeric fields available for visualization.")

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR2 dataset was loaded from its Croissant schema and inspected using the `mlcroissant` library.
- Record sets, fields, and sample records were previewed by referencing their `@id` values.
- Exploratory steps included basic analysis of numeric fields and visualizing their distributions for further insight.
- For deeper biological or domain analysis, enrich the notebook with domain-specific feature engineering or link field `@id`s to curated bioinformatics resources.
